In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q transformers datasets accelerate evaluate wandb

In [3]:
!pip install -U torchvision

In [4]:
!pip uninstall -y transformers datasets torchvision

!pip install -q \
transformers==4.52.4 \
datasets==3.6.0 \
accelerate==1.7.0 \
evaluate==0.4.3 \
wandb

Found existing installation: transformers 5.10.2
Uninstalling transformers-5.10.2:
  Successfully uninstalled transformers-5.10.2
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follo

In [5]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import evaluate
import wandb

In [6]:
wandb.login()

wandb.init(
    project="23f1000054-t22026",
    name="deberta_v3_small_v1"
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [7]:
train = pd.read_csv(
    "/content/drive/MyDrive/SmartMCQ/train.csv"
)

test = pd.read_csv(
    "/content/drive/MyDrive/SmartMCQ/test.csv"
)

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [8]:
rows = []

for _, row in train.iterrows():

    prompt = row["prompt"]
    answer = row["answer"]

    for option in ["A","B","C","D","E"]:

        rows.append({
            "text": prompt,
            "option": str(row[option]),
            "label": int(option == answer)
        })

binary_train = pd.DataFrame(rows)

print(binary_train.shape)
binary_train.head()

(10000, 3)


,text,option,label
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,0
1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,1
2,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,0
3,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,0
4,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,0


In [9]:
train_df, valid_df = train_test_split(
    binary_train,
    test_size=0.2,
    random_state=42,
    stratify=binary_train["label"]
)

print(train_df.shape)
print(valid_df.shape)

(8000, 3)
(2000, 3)


In [10]:
train_ds = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

valid_ds = Dataset.from_pandas(
    valid_df.reset_index(drop=True)
)

In [11]:
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
 

In [12]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        batch["option"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

valid_ds = valid_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [13]:
train_ds = train_ds.remove_columns(
    ["text","option"]
)

valid_ds = valid_ds.remove_columns(
    ["text","option"]
)

train_ds.set_format("torch")
valid_ds.set_format("torch")

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(
        logits,
        axis=1
    )
    acc = accuracy_score(
        labels,
        preds
    )
    f1 = f1_score(
        labels,
        preds
    )
    return {
        "accuracy": acc,
        "f1": f1
    }

In [16]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="wandb"
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics
)

In [18]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.496600,0.460348,0.800000,0.000000
2,0.342200,0.267623,0.906500,0.717949
3,0.223100,0.248770,0.929500,0.798283


TrainOutput(global_step=1500, training_loss=0.3539689229329427, metrics={'train_runtime': 467.8793, 'train_samples_per_second': 51.295, 'train_steps_per_second': 3.206, 'total_flos': 1589665406976000.0, 'train_loss': 0.3539689229329427, 'epoch': 3.0})

In [20]:
trainer.evaluate()

{'eval_loss': 0.24876965582370758,
 'eval_accuracy': 0.9295,
 'eval_f1': 0.7982832618025751,
 'eval_runtime': 11.0074,
 'eval_samples_per_second': 181.697,
 'eval_steps_per_second': 11.356,
 'epoch': 3.0}

In [21]:
trainer.save_model(
    "/content/drive/MyDrive/SmartMCQ/deberta_v3_small"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/SmartMCQ/deberta_v3_small"
)

('/content/drive/MyDrive/SmartMCQ/deberta_v3_small/tokenizer_config.json',
 '/content/drive/MyDrive/SmartMCQ/deberta_v3_small/special_tokens_map.json',
 '/content/drive/MyDrive/SmartMCQ/deberta_v3_small/spm.model',
 '/content/drive/MyDrive/SmartMCQ/deberta_v3_small/added_tokens.json',
 '/content/drive/MyDrive/SmartMCQ/deberta_v3_small/tokenizer.json')

In [22]:
wandb.finish()

eval/accuracy,▁▇██
eval/f1,▁▇██
eval/loss,█▂▁▁
eval/runtime,█▃▄▁
eval/samples_per_second,▁▆▅█
eval/steps_per_second,▁▆▅█
train/epoch,▁▁▅▅████
train/global_step,▁▁▅▅████
train/grad_norm,▁▆█
train/learning_rate,█▄▁
+1,...
